In [62]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score,cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, f1_score

In [63]:

NAME = f"Daniel_Tesfai_Kebede" 
MATRIKELNUMMER = "1716694"

TRAIN_PATH = "train.csv"
HOLDBACK_PATH = "holdback.csv"

OUT_PATH = f"predictions_{NAME}_{MATRIKELNUMMER}.csv"
OUT_PATH

'predictions_Daniel_Tesfai_Kebede_1716694.csv'

In [64]:
# Load training data
train_df = pd.read_csv(TRAIN_PATH)

# Show basic info
print("train shape:", train_df.shape)
print("train columns (raw):", list(train_df.columns))
display(train_df.head(3))

# ---- FIX COLUMN NAMES (explicit, no guessing) ----
# Your file uses 'targets' instead of 'label'
train_df = train_df.rename(columns={"targets": "label"})

# Verify after rename
print("\ntrain columns (after rename):", list(train_df.columns))
display(train_df.head(3))

# ---- HARD ASSERTIONS (must pass) ----
required_train_cols = {"id", "samples", "label"}
assert required_train_cols.issubset(train_df.columns), (
    f"train.csv must contain exactly these logical columns: {required_train_cols}"
)

# Label sanity check
assert train_df["label"].isin([0, 1, 2]).all(), "Labels must be exactly 0, 1, or 2"

print("\nLabel distribution:")
print(train_df["label"].value_counts().sort_index())

train shape: (950, 3)
train columns (raw): ['id', 'samples', 'targets']


,id,samples,targets
0,0,"@AoDespair In hindsight, the craziest thing ab...",0
1,1,@catturd2 BREAKING... \r\n\r\nMerrick Garland ...,1
2,2,@cmclymer It was a very well thought out plan.,1



train columns (after rename): ['id', 'samples', 'label']


,id,samples,label
0,0,"@AoDespair In hindsight, the craziest thing ab...",0
1,1,@catturd2 BREAKING... \r\n\r\nMerrick Garland ...,1
2,2,@cmclymer It was a very well thought out plan.,1



Label distribution:
label
0    321
1    334
2    295
Name: count, dtype: int64


In [65]:
def clean_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s)
    s = s.replace("\u200b", " ")         # zero-width space
    s = re.sub(r"\s+", " ", s).strip()   # normalize whitespace
    return s

train_df["samples_clean"] = train_df["samples"].map(clean_text)
train_df[["samples", "samples_clean"]].head()

,samples,samples_clean
0,"@AoDespair In hindsight, the craziest thing ab...","@AoDespair In hindsight, the craziest thing ab..."
1,@catturd2 BREAKING... \r\n\r\nMerrick Garland ...,@catturd2 BREAKING... Merrick Garland has appo...
2,@cmclymer It was a very well thought out plan.,@cmclymer It was a very well thought out plan.
3,@aintscarylarry He's crying because he's a loser.,@aintscarylarry He's crying because he's a loser.
4,@mtgreenee You’re right — the people have the ...,@mtgreenee You’re right — the people have the ...


In [66]:
from sklearn.pipeline import FeatureUnion

model = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            strip_accents="unicode"
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        ))
    ])),
    ("clf", LinearSVC(C=1.0, class_weight="balanced"))
])

X = train_df["samples_clean"].values
y = train_df["label"].values

In [67]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro")
print("F1_macro per fold:", np.round(scores, 4))
print("Mean F1_macro:", scores.mean().round(4), "+/-", scores.std().round(4))

F1_macro per fold: [0.5885 0.6118 0.5974 0.6436 0.6092]
Mean F1_macro: 0.6101 +/- 0.0187


In [68]:
y_pred = cross_val_predict(model, X, y, cv=cv)

print("Macro F1:", round(f1_score(y, y_pred, average="macro"), 4))
print("\nClassification report:\n", classification_report(y, y_pred, digits=4))
print("\nConfusion matrix:\n", confusion_matrix(y, y_pred))

Macro F1: 0.6117

Classification report:
               precision    recall  f1-score   support

           0     0.5260    0.5047    0.5151       321
           1     0.5573    0.5389    0.5479       334
           2     0.7429    0.8034    0.7720       295

    accuracy                         0.6095       950
   macro avg     0.6087    0.6157    0.6117       950
weighted avg     0.6044    0.6095    0.6064       950


Confusion matrix:
 [[162 116  43]
 [115 180  39]
 [ 31  27 237]]


In [69]:
model.fit(X, y)
print("Trained final model on full train.csv ✅")

Trained final model on full train.csv ✅


In [70]:
if not os.path.exists(HOLDBACK_PATH):
    print("holdback.csv not found yet ✅")
    print("That's expected if they haven't released it.")
    print("When you receive holdback.csv, put it next to this notebook and rerun Cells 9–11.")
else:
    hold_df = pd.read_csv(HOLDBACK_PATH)

    required_hold_cols = {"id", "samples"}
    assert required_hold_cols.issubset(hold_df.columns), f"holdback.csv must contain columns: {required_hold_cols}"

    hold_df["samples_clean"] = hold_df["samples"].map(clean_text)

    hold_X = hold_df["samples_clean"].values
    hold_pred = model.predict(hold_X)

    pred_df = pd.DataFrame({
        "id": hold_df["id"].values,
        "label": hold_pred.astype(int)
    })

    pred_df.to_csv(OUT_PATH, index=False)
    print("Wrote predictions file ✅:", OUT_PATH)
    display(pred_df.head())

holdback.csv not found yet ✅
That's expected if they haven't released it.
When you receive holdback.csv, put it next to this notebook and rerun Cells 9–11.


In [71]:
if "pred_df" not in globals():
    print("No predictions created yet (no holdback.csv) ✅")
else:
    assert list(pred_df.columns) == ["id", "label"], "Output must have exactly columns: id,label"
    assert pred_df["id"].isna().sum() == 0, "No missing ids allowed"
    assert pred_df["label"].isin([0, 1, 2]).all(), "Labels must be 0/1/2 only"
    assert pred_df["id"].is_unique, "Each id should appear once"
    print("Local format checks passed ✅")

No predictions created yet (no holdback.csv) ✅


In [72]:
checker = "format_checker.py"

if "OUT_PATH" not in globals() or not os.path.exists(OUT_PATH):
    print("No predictions file yet → cannot run format checker ✅")
elif not os.path.exists(checker):
    print("format_checker.py not found in folder (skip).")
else:
    import subprocess, sys
    result = subprocess.run([sys.executable, checker, OUT_PATH], capture_output=True, text=True)
    print("Checker STDOUT:\n", result.stdout)
    print("Checker STDERR:\n", result.stderr)
    print("Return code:", result.returncode)

No predictions file yet → cannot run format checker ✅
